In [66]:
sc

<SparkContext master=local[*] appName=MyCustomApp>

In [4]:
from pyspark import SparkContext

# 1. Force stop the hidden background context that is blocking you
SparkContext.setSystemProperty('spark.driver.allowMultipleContexts', 'false')
if 'sc' in locals() or 'sc' in globals():
    try:
        sc.stop()
    except:
        pass

# 2. Now you can safely initialize your custom app
sc = SparkContext(master="local[*]", appName="MyCustomApp")

# 3. Verify it worked
print(f"Success! Active App Name: {sc.appName}")


Success! Active App Name: MyCustomApp


In [63]:
sc.parallelize("a b c d").flatMap(lambda x:x.split(" ")[0]).take(4)

['a', 'b', 'c', 'd']

In [6]:
!pwd

/home/student/LAB/ApacheSpark


In [7]:
log = sc.wholeTextFiles("./spark.log")

In [8]:
lines = sc.textFile("./spark.log")

In [65]:
lines

file:///home/student/LAB/ApacheSpark/spark.log/ MapPartitionsRDD[15] at textFile at NativeMethodAccessorImpl.java:0

### Map and Reduce Example

In [18]:
lines = sc.textFile("file:///home/student/LAB/ApacheSpark/spark.log/")

In [19]:
lines.take(10)

['17/06/09 20:10:40 INFO executor.CoarseGrainedExecutorBackend: Registered signal handlers for [TERM, HUP, INT]',
 '17/06/09 20:10:40 INFO spark.SecurityManager: Changing view acls to: yarn,curi',
 '17/06/09 20:10:40 INFO spark.SecurityManager: Changing modify acls to: yarn,curi',
 '17/06/09 20:10:40 INFO spark.SecurityManager: SecurityManager: authentication disabled; ui acls disabled; users with view permissions: Set(yarn, curi); users with modify permissions: Set(yarn, curi)',
 '17/06/09 20:10:41 INFO spark.SecurityManager: Changing view acls to: yarn,curi',
 '17/06/09 20:10:41 INFO spark.SecurityManager: Changing modify acls to: yarn,curi',
 '17/06/09 20:10:41 INFO spark.SecurityManager: SecurityManager: authentication disabled; ui acls disabled; users with view permissions: Set(yarn, curi); users with modify permissions: Set(yarn, curi)',
 '17/06/09 20:10:41 INFO slf4j.Slf4jLogger: Slf4jLogger started',
 '17/06/09 20:10:41 INFO Remoting: Starting remoting',
 '17/06/09 20:10:41 INF

In [20]:
lineLengths = lines.map(lambda s: len(s))
lineLengths.take(10)

[109, 78, 80, 198, 78, 80, 198, 61, 50, 133]

In [21]:
totalLength = lineLengths.reduce(lambda a,b: a+b)
print(totalLength)

192268


### Lab 1 : Log Analytic with RDDs

In [22]:
log = sc.textFile("file:////home/student/LAB/ApacheSpark/spark.log/")

In [23]:
log.collect()

['17/06/09 20:10:40 INFO executor.CoarseGrainedExecutorBackend: Registered signal handlers for [TERM, HUP, INT]',
 '17/06/09 20:10:40 INFO spark.SecurityManager: Changing view acls to: yarn,curi',
 '17/06/09 20:10:40 INFO spark.SecurityManager: Changing modify acls to: yarn,curi',
 '17/06/09 20:10:40 INFO spark.SecurityManager: SecurityManager: authentication disabled; ui acls disabled; users with view permissions: Set(yarn, curi); users with modify permissions: Set(yarn, curi)',
 '17/06/09 20:10:41 INFO spark.SecurityManager: Changing view acls to: yarn,curi',
 '17/06/09 20:10:41 INFO spark.SecurityManager: Changing modify acls to: yarn,curi',
 '17/06/09 20:10:41 INFO spark.SecurityManager: SecurityManager: authentication disabled; ui acls disabled; users with view permissions: Set(yarn, curi); users with modify permissions: Set(yarn, curi)',
 '17/06/09 20:10:41 INFO slf4j.Slf4jLogger: Slf4jLogger started',
 '17/06/09 20:10:41 INFO Remoting: Starting remoting',
 '17/06/09 20:10:41 INF

In [24]:
log.count()

2000

In [25]:
log.countApprox(100)

2000

In [26]:
date = log.map(lambda x: x.split(' ')[0])

In [27]:
date.take(3)

['17/06/09', '17/06/09', '17/06/09']

In [28]:
print(log.flatMap(lambda x: x.split()).take(15))

['17/06/09', '20:10:40', 'INFO', 'executor.CoarseGrainedExecutorBackend:', 'Registered', 'signal', 'handlers', 'for', '[TERM,', 'HUP,', 'INT]', '17/06/09', '20:10:40', 'INFO', 'spark.SecurityManager:']


In [29]:
log.take(3)

['17/06/09 20:10:40 INFO executor.CoarseGrainedExecutorBackend: Registered signal handlers for [TERM, HUP, INT]',
 '17/06/09 20:10:40 INFO spark.SecurityManager: Changing view acls to: yarn,curi',
 '17/06/09 20:10:40 INFO spark.SecurityManager: Changing modify acls to: yarn,curi']

In [32]:
source_info_rdd = log.map(lambda x: (x.split()[3]," ".join(x.split()[4:])))

In [33]:
source_info_rdd.take(3)

[('executor.CoarseGrainedExecutorBackend:',
  'Registered signal handlers for [TERM, HUP, INT]'),
 ('spark.SecurityManager:', 'Changing view acls to: yarn,curi'),
 ('spark.SecurityManager:', 'Changing modify acls to: yarn,curi')]

In [34]:
source_info_rdd.keys().take(3)

['executor.CoarseGrainedExecutorBackend:',
 'spark.SecurityManager:',
 'spark.SecurityManager:']

In [35]:
source_info_rdd.values().take(3)

['Registered signal handlers for [TERM, HUP, INT]',
 'Changing view acls to: yarn,curi',
 'Changing modify acls to: yarn,curi']

In [36]:
source_len_rdd = source_info_rdd.mapValues(lambda x: len(x.split()))

In [37]:
source_len_rdd.take(4)

[('executor.CoarseGrainedExecutorBackend:', 7),
 ('spark.SecurityManager:', 5),
 ('spark.SecurityManager:', 5),
 ('spark.SecurityManager:', 18)]

In [38]:
source_word_rdd = source_info_rdd.flatMapValues(lambda x: x.split())

In [39]:
source_word_rdd.take(10)

[('executor.CoarseGrainedExecutorBackend:', 'Registered'),
 ('executor.CoarseGrainedExecutorBackend:', 'signal'),
 ('executor.CoarseGrainedExecutorBackend:', 'handlers'),
 ('executor.CoarseGrainedExecutorBackend:', 'for'),
 ('executor.CoarseGrainedExecutorBackend:', '[TERM,'),
 ('executor.CoarseGrainedExecutorBackend:', 'HUP,'),
 ('executor.CoarseGrainedExecutorBackend:', 'INT]'),
 ('spark.SecurityManager:', 'Changing'),
 ('spark.SecurityManager:', 'view'),
 ('spark.SecurityManager:', 'acls')]

In [40]:
actor_log_rdd = source_info_rdd.mapValues(lambda x:x.split())

In [41]:
actor_log_rdd.take(4)

[('executor.CoarseGrainedExecutorBackend:',
  ['Registered', 'signal', 'handlers', 'for', '[TERM,', 'HUP,', 'INT]']),
 ('spark.SecurityManager:', ['Changing', 'view', 'acls', 'to:', 'yarn,curi']),
 ('spark.SecurityManager:',
  ['Changing', 'modify', 'acls', 'to:', 'yarn,curi']),
 ('spark.SecurityManager:',
  ['SecurityManager:',
   'authentication',
   'disabled;',
   'ui',
   'acls',
   'disabled;',
   'users',
   'with',
   'view',
   'permissions:',
   'Set(yarn,',
   'curi);',
   'users',
   'with',
   'modify',
   'permissions:',
   'Set(yarn,',
   'curi)'])]

In [42]:
filtered_rdd=actor_log_rdd.filter(lambda x: x[0] == 'spark.SecurityManager:')

In [43]:
filtered_rdd.take(4)

[('spark.SecurityManager:', ['Changing', 'view', 'acls', 'to:', 'yarn,curi']),
 ('spark.SecurityManager:',
  ['Changing', 'modify', 'acls', 'to:', 'yarn,curi']),
 ('spark.SecurityManager:',
  ['SecurityManager:',
   'authentication',
   'disabled;',
   'ui',
   'acls',
   'disabled;',
   'users',
   'with',
   'view',
   'permissions:',
   'Set(yarn,',
   'curi);',
   'users',
   'with',
   'modify',
   'permissions:',
   'Set(yarn,',
   'curi)']),
 ('spark.SecurityManager:', ['Changing', 'view', 'acls', 'to:', 'yarn,curi'])]

In [44]:
source_len_rdd = source_info_rdd.map(lambda x: (x[0], len(x[1].split())))
source_len_rdd.take(4)

[('executor.CoarseGrainedExecutorBackend:', 7),
 ('spark.SecurityManager:', 5),
 ('spark.SecurityManager:', 5),
 ('spark.SecurityManager:', 18)]

In [45]:
len_rdd=source_len_rdd.values()
len_rdd.take(4)

[7, 5, 5, 18]

In [46]:
len_rdd.reduce(lambda x,y:x+y)

17511

In [47]:
source_nword_rdd = source_len_rdd.reduceByKey(lambda x,y: x+y)
source_nword_rdd.take(10)

[('spark.SecurityManager:', 56),
 ('Remoting:', 8),
 ('storage.DiskBlockManager:', 5),
 ('storage.MemoryStore:', 2092),
 ('executor.Executor:', 6647),
 ('broadcast.TorrentBroadcast:', 444),
 ('Configuration.deprecation:', 30),
 ('output.FileOutputCommitter:', 420),
 ('mapred.SparkHadoopMapRedUtil:', 60),
 ('executor.CoarseGrainedExecutorBackend:', 1235)]

In [48]:
source_nline_rdd = source_len_rdd.groupByKey().mapValues(lambda x: len(x))
source_nline_rdd.take(10)

[('spark.SecurityManager:', 6),
 ('Remoting:', 2),
 ('storage.DiskBlockManager:', 1),
 ('storage.MemoryStore:', 150),
 ('executor.Executor:', 606),
 ('broadcast.TorrentBroadcast:', 74),
 ('Configuration.deprecation:', 5),
 ('output.FileOutputCommitter:', 60),
 ('mapred.SparkHadoopMapRedUtil:', 30),
 ('executor.CoarseGrainedExecutorBackend:', 308)]

In [49]:
join_result = source_nline_rdd.join(source_nword_rdd)
join_result.take(10)

[('spark.SecurityManager:', (6, 56)),
 ('Remoting:', (2, 8)),
 ('storage.DiskBlockManager:', (1, 5)),
 ('storage.MemoryStore:', (150, 2092)),
 ('executor.Executor:', (606, 6647)),
 ('broadcast.TorrentBroadcast:', (74, 444)),
 ('Configuration.deprecation:', (5, 30)),
 ('output.FileOutputCommitter:', (60, 420)),
 ('mapred.SparkHadoopMapRedUtil:', (30, 60)),
 ('executor.CoarseGrainedExecutorBackend:', (308, 1235))]

In [50]:
source_len_rdd.countByKey()

defaultdict(int,
            {'executor.CoarseGrainedExecutorBackend:': 308,
             'spark.SecurityManager:': 6,
             'slf4j.Slf4jLogger:': 1,
             'Remoting:': 2,
             'util.Utils:': 2,
             'storage.DiskBlockManager:': 1,
             'storage.MemoryStore:': 150,
             'executor.Executor:': 606,
             'netty.NettyBlockTransferService:': 1,
             'storage.BlockManagerMaster:': 2,
             'broadcast.TorrentBroadcast:': 74,
             'spark.CacheManager:': 75,
             'rdd.HadoopRDD:': 45,
             'Configuration.deprecation:': 5,
             'python.PythonRunner:': 375,
             'storage.BlockManager:': 257,
             'output.FileOutputCommitter:': 60,
             'mapred.SparkHadoopMapRedUtil:': 30})

In [51]:
source_len_rdd.countByValue()

defaultdict(int,
            {('executor.CoarseGrainedExecutorBackend:', 7): 1,
             ('spark.SecurityManager:', 5): 4,
             ('spark.SecurityManager:', 18): 2,
             ('slf4j.Slf4jLogger:', 2): 1,
             ('Remoting:', 2): 1,
             ('Remoting:', 6): 1,
             ('util.Utils:', 7): 2,
             ('storage.DiskBlockManager:', 5): 1,
             ('storage.MemoryStore:', 6): 1,
             ('executor.CoarseGrainedExecutorBackend:', 4): 307,
             ('executor.Executor:', 7): 1,
             ('netty.NettyBlockTransferService:', 4): 1,
             ('storage.BlockManagerMaster:', 4): 1,
             ('storage.BlockManagerMaster:', 2): 1,
             ('executor.Executor:', 8): 305,
             ('broadcast.TorrentBroadcast:', 5): 37,
             ('storage.MemoryStore:', 14): 149,
             ('broadcast.TorrentBroadcast:', 7): 37,
             ('spark.CacheManager:', 6): 75,
             ('rdd.HadoopRDD:', 3): 45,
             ('Configuration.d

In [52]:
source_rdd = source_info_rdd.keys()
source_rdd.take(3)

['executor.CoarseGrainedExecutorBackend:',
 'spark.SecurityManager:',
 'spark.SecurityManager:']

In [53]:
uniq_source_rdd = source_rdd.distinct()
uniq_source_rdd.take(100)

['spark.SecurityManager:',
 'Remoting:',
 'storage.DiskBlockManager:',
 'storage.MemoryStore:',
 'executor.Executor:',
 'broadcast.TorrentBroadcast:',
 'Configuration.deprecation:',
 'output.FileOutputCommitter:',
 'mapred.SparkHadoopMapRedUtil:',
 'executor.CoarseGrainedExecutorBackend:',
 'slf4j.Slf4jLogger:',
 'util.Utils:',
 'netty.NettyBlockTransferService:',
 'storage.BlockManagerMaster:',
 'spark.CacheManager:',
 'rdd.HadoopRDD:',
 'python.PythonRunner:',
 'storage.BlockManager:']

In [54]:
source_avglen_rdd = source_len_rdd.groupByKey().mapValues(lambda x:sum(x)/len(x))

In [55]:
source_avglen_rdd.sortBy(lambda x: x[1]).take(10)

[('mapred.SparkHadoopMapRedUtil:', 2.0),
 ('slf4j.Slf4jLogger:', 2.0),
 ('storage.BlockManagerMaster:', 3.0),
 ('rdd.HadoopRDD:', 3.0),
 ('Remoting:', 4.0),
 ('netty.NettyBlockTransferService:', 4.0),
 ('storage.BlockManager:', 4.0),
 ('executor.CoarseGrainedExecutorBackend:', 4.009740259740259),
 ('storage.DiskBlockManager:', 5.0),
 ('broadcast.TorrentBroadcast:', 6.0)]

In [56]:
source_avglen_rdd.sortByKey().take(10)

[('Configuration.deprecation:', 6.0),
 ('Remoting:', 4.0),
 ('broadcast.TorrentBroadcast:', 6.0),
 ('executor.CoarseGrainedExecutorBackend:', 4.009740259740259),
 ('executor.Executor:', 10.968646864686468),
 ('mapred.SparkHadoopMapRedUtil:', 2.0),
 ('netty.NettyBlockTransferService:', 4.0),
 ('output.FileOutputCommitter:', 7.0),
 ('python.PythonRunner:', 13.0),
 ('rdd.HadoopRDD:', 3.0)]

In [57]:
source_avglen_rdd.sortBy(lambda x: x[0]).take(10)

[('Configuration.deprecation:', 6.0),
 ('Remoting:', 4.0),
 ('broadcast.TorrentBroadcast:', 6.0),
 ('executor.CoarseGrainedExecutorBackend:', 4.009740259740259),
 ('executor.Executor:', 10.968646864686468),
 ('mapred.SparkHadoopMapRedUtil:', 2.0),
 ('netty.NettyBlockTransferService:', 4.0),
 ('output.FileOutputCommitter:', 7.0),
 ('python.PythonRunner:', 13.0),
 ('rdd.HadoopRDD:', 3.0)]

In [58]:
source_avglen_rdd.getNumPartitions()

2

In [59]:
source_avglen_rdd = source_avglen_rdd.repartition(2)

In [60]:
source_avglen_rdd.getNumPartitions()

2

In [61]:
source_len_rdd.take(4)

[('executor.CoarseGrainedExecutorBackend:', 7),
 ('spark.SecurityManager:', 5),
 ('spark.SecurityManager:', 5),
 ('spark.SecurityManager:', 18)]

In [62]:
source_len_rdd.values().mean()

8.755500000000007